# 09.9 · Calling LLMs from Python

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/09-nlp-genai/09-calling-llms-from-python/lesson.ipynb)

Read the [README](README.md) first. Here we make the same request three ways (the Anthropic SDK, the OpenAI client pointed at Ollama, and Ollama's own API), then look inside the course helper, and finally measure tokens, cost and latency.

> **About the outputs in this notebook.** They were produced with `llama3.2` (3B) via Ollama on a laptop. Yours will differ: a different model, a different run, even the same model on a different machine. Small local models make more mistakes than large hosted ones; that is part of what this lesson shows.

In [1]:
# --- course files: found locally, downloaded on Colab -------------------------
import sys, urllib.request
from pathlib import Path
REPO_RAW = "https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/"

def repo_file(rel):
    """Return a local path to a file in this repo, downloading it if needed."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / rel).exists():
            return parent / rel
    dest = Path("course_files") / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    if not dest.exists():
        urllib.request.urlretrieve(REPO_RAW + rel, dest)
    return dest

In [2]:
sys.path.insert(0, str(repo_file("lib/course_llm.py").parent))
import course_llm
from course_llm import ask, ask_json, chat, info
info();

LLM provider: ollama · model: llama3.2


`info()` also loaded your `.env` file into the environment, which the raw calls below rely on. Which keys are available? We print **only whether** each key is set, never the key itself.

In [3]:
import os
for var in ["ANTHROPIC_API_KEY", "OPENAI_API_KEY", "GEMINI_API_KEY", "GROQ_API_KEY"]:
    print(f"{var:18} {'set' if os.environ.get(var) else 'not set'}")
print("Ollama running:   ", course_llm._ollama_running())

ANTHROPIC_API_KEY  not set
OPENAI_API_KEY     not set
GEMINI_API_KEY     not set
GROQ_API_KEY       not set
Ollama running:    True


## 1. The raw calls

All three send the same thing: a system prompt and one user message. All three return text plus token counts. Only the spelling differs.

### Anthropic SDK (Claude)

Runs only if `ANTHROPIC_API_KEY` is set. The client reads the key from the environment, so it never appears in your code.

In [4]:
SYSTEM = "You are a concise assistant for an online gift shop."
QUESTION = "In one sentence, what is customer churn?"

if os.environ.get("ANTHROPIC_API_KEY"):
    import anthropic
    client = anthropic.Anthropic()                       # key comes from the environment
    response = client.messages.create(model="claude-opus-5-5", max_tokens=16000,
                                      system=SYSTEM,
                                      messages=[{"role": "user", "content": QUESTION}])
    print("".join(block.text for block in response.content if block.type == "text"))
    print("tokens in/out:", response.usage.input_tokens, response.usage.output_tokens)
else:
    print("Skipped: add ANTHROPIC_API_KEY to .env to run this cell.")

Skipped: add ANTHROPIC_API_KEY to .env to run this cell.


Notes: the system prompt is a separate `system=` argument; the reply is a list of **content blocks** (text, and on current models also thinking blocks), so we join the text ones.

### OpenAI client, pointed at Ollama

The OpenAI chat format has become a common standard: OpenAI, Gemini, Groq, LM Studio and Ollama all accept it. Only `base_url`, the key and the model name change.

In [5]:
if course_llm._ollama_running():
    from openai import OpenAI
    client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")  # Ollama ignores the key
    response = client.chat.completions.create(
        model="llama3.2", max_tokens=200,
        messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": QUESTION}])
    print(response.choices[0].message.content)
    print("tokens in/out:", response.usage.prompt_tokens, response.usage.completion_tokens)
else:
    print("Skipped: Ollama is not running.")

Customer churn refers to the percentage of customers who stop buying from a business or cancel their subscription over a specific period, resulting in a loss of sales and revenue.
tokens in/out: 45 33


Here the system prompt is just the first message, with `role: "system"`. The reply is `choices[0].message.content`.

### Ollama's own API

Plain HTTP and JSON, no client library at all. This is what the course helper uses for Ollama, because it is the only way to set `num_ctx`, the context window that decides how much memory the model takes (see [setup](../../../setup/README.md)).

In [6]:
import json, urllib.request

if course_llm._ollama_running():
    body = {"model": "llama3.2", "stream": False, "options": {"num_ctx": 8192},
            "messages": [{"role": "system", "content": SYSTEM}, {"role": "user", "content": QUESTION}]}
    request = urllib.request.Request("http://localhost:11434/api/chat", json.dumps(body).encode(),
                                     {"Content-Type": "application/json"})
    data = json.load(urllib.request.urlopen(request))
    print(data["message"]["content"])
    print("tokens in/out:", data["prompt_eval_count"], data["eval_count"])
else:
    print("Skipped: Ollama is not running.")

Customer churn refers to the percentage of customers who stop doing business with a company over a specific period of time, often resulting in lost sales and revenue.
tokens in/out: 45 31


## 2. Why the course has a helper

Three spellings of one idea. Writing all three in every notebook would bury the lesson under plumbing, and would lock each notebook to one provider. So `lib/course_llm.py` wraps them: `ask`, `ask_json` and `chat` look the same whichever provider you chose in `.env`.

It is short enough to read. Here is how it picks a provider:

In [7]:
import inspect
print(inspect.getsource(course_llm._choose_provider))

def _choose_provider() -> str:
    _load_dotenv()
    chosen = os.environ.get("LLM_PROVIDER", "").strip().lower()
    if chosen:
        if chosen != "anthropic" and chosen not in OPENAI_COMPATIBLE:
            raise ValueError(f"Unknown LLM_PROVIDER '{chosen}'. "
                             f"Use one of: anthropic, {', '.join(OPENAI_COMPATIBLE)}")
        return chosen
    if os.environ.get("ANTHROPIC_API_KEY"):
        return "anthropic"
    if os.environ.get("OPENAI_API_KEY"):
        return "openai"
    if _ollama_running():
        return "ollama"
    raise RuntimeError(
        "No LLM available. Pick one:\n"
        "  free, local : install Ollama (https://ollama.com), run `ollama pull llama3.2`\n"
        "  Claude      : add ANTHROPIC_API_KEY=... to .env\n"
        "  OpenAI      : add OPENAI_API_KEY=... to .env\n"
        "  others      : set LLM_PROVIDER=groq|gemini|lmstudio and its key (see setup/README.md)"
    )



And here is the whole Claude path. Compare it with the raw call above: same SDK, same `messages.create` shape. Two additions: structured output via `output_config`, and a **fallback**: if a current Claude model declines a request, the API retries it on another Claude model instead of failing.

In [8]:
print(inspect.getsource(course_llm._ask_anthropic))

def _ask_anthropic(messages, system, max_tokens, schema):
    import anthropic
    client = anthropic.Anthropic()
    kwargs = dict(model=MODEL, max_tokens=max_tokens, messages=messages)
    if system:
        kwargs["system"] = system
    if schema:
        kwargs["output_config"] = {"format": {"type": "json_schema", "schema": schema}}
    # If the model declines a request, the API retries it on a fallback model.
    response = client.beta.messages.create(
        **kwargs, betas=["server-side-fallback-2026-07-01"], fallbacks="default")
    if response.stop_reason == "refusal":
        raise RuntimeError("The model declined this request.")
    last_usage.update(input=response.usage.input_tokens, output=response.usage.output_tokens)
    return "".join(block.text for block in response.content if block.type == "text")



In production, teams often use a library such as **LiteLLM**, which does the same job for 100+ providers. The course writes its own small helper (about 240 lines, half of them comments) so you can read every line.

## 3. Tokens and cost

You pay per token, separately for input (your prompt) and output (the answer). Let us measure a real task: classify five product reviews.

In [9]:
REVIEWS = [
    "The ceramic mug arrived chipped and support never replied to my email.",
    "Lovely tea towels, bright colours, washed well. Will buy again!",
    "The lantern is fine. Nothing special, but it does the job.",
    "Absolutely gorgeous fairy lights, my daughter loved them.",
    "Parcel came two weeks late and the cake stand was the wrong colour.",
]
PROMPT = ("Classify the sentiment of this product review as positive, neutral or negative, "
          "and name the product. Reply as: sentiment, product.\nReview: ")

In [10]:
import time
records = []
for review in REVIEWS:
    start = time.time()
    answer = ask(PROMPT + review, temperature=0, max_tokens=30)
    records.append((course_llm.last_usage["input"], course_llm.last_usage["output"],
                    time.time() - start, answer.strip()))
for tokens_in, tokens_out, secs, answer in records:
    print(f"{tokens_in:4} in {tokens_out:3} out {secs:5.2f} s  {answer[:60]}")

  68 in   5 out  0.13 s  Negative, Coffee Mug
  68 in   6 out  0.12 s  Positive, Tea Towel
  68 in   6 out  0.12 s  Neutral, String Lights.
  64 in   6 out  0.12 s  Positive, Fairy Lights.
  68 in   5 out  0.12 s  Negative, Cake Stand


In [11]:
avg_in = sum(r[0] for r in records) / len(records)
avg_out = sum(r[1] for r in records) / len(records)
print(f"average per review: {avg_in:.0f} input tokens, {avg_out:.0f} output tokens")

average per review: 67 input tokens, 6 output tokens


Token counts depend on the model's tokenizer and on the chat template the provider wraps around your message, so a hosted model will report slightly different numbers. Now scale it to 1,000 reviews:

In [12]:
# US dollars per million tokens (input, output). Prices change: check the provider's
# pricing page before you rely on these. Claude prices as published in October 2026.
PRICES = {
    "Claude Haiku 5.5":  (0.10, 0.50),
    "Claude Sonnet 5.5": (2.00, 10.00),
    "Claude Opus 5.5":   (4.00, 20.00),
    "Ollama (local)":    (0.00, 0.00),     # you pay in hardware and electricity instead
}

def cost(n_requests, tokens_in, tokens_out, price_in, price_out):
    return n_requests * (tokens_in * price_in + tokens_out * price_out) / 1_000_000

In [13]:
for name, (p_in, p_out) in PRICES.items():
    print(f"{name:18} ${cost(1000, avg_in, avg_out, p_in, p_out):7.3f} for 1,000 reviews")
print()
print("Same task, 1,000 reviews, with a 400-token few-shot system prompt added to every request:")
for name, (p_in, p_out) in PRICES.items():
    print(f"{name:18} ${cost(1000, avg_in + 400, avg_out, p_in, p_out):7.3f}")

Claude Haiku 5.5   $  0.010 for 1,000 reviews
Claude Sonnet 5.5  $  0.190 for 1,000 reviews
Claude Opus 5.5    $  0.381 for 1,000 reviews
Ollama (local)     $  0.000 for 1,000 reviews

Same task, 1,000 reviews, with a 400-token few-shot system prompt added to every request:
Claude Haiku 5.5   $  0.050
Claude Sonnet 5.5  $  0.990
Claude Opus 5.5    $  1.981
Ollama (local)     $  0.000


Two lessons. Output tokens cost 5× input tokens here, so ask for short answers. And a long system prompt is paid for **on every request**: the few-shot prompt that bought accuracy in 09.8 multiplies the input bill. (Providers offer **prompt caching** to discount a repeated prefix; see their docs.)

## 4. Latency

In [14]:
times = sorted(r[2] for r in records)
print(f"5 calls: fastest {times[0]:.2f} s, median {times[2]:.2f} s, slowest {times[-1]:.2f} s")
print(f"1,000 reviews one after another ≈ {sum(times) / len(times) * 1000 / 60:.1f} minutes")

5 calls: fastest 0.12 s, median 0.12 s, slowest 0.13 s
1,000 reviews one after another ≈ 2.0 minutes


The first call to a local model includes loading it into memory, so it is usually the slowest. Hosted APIs add network time but can run many requests in parallel; providers also offer **batch** APIs that are cheaper when you do not need answers immediately.

## 5. When calls fail

Networks drop, providers rate-limit, models time out. The standard answer is to **retry with exponential backoff**: wait 1 s, then 2 s, then 4 s. (The Anthropic and OpenAI SDKs already retry some errors on their own.) To see it work, we point the helper at a model that does not exist.

In [15]:
def ask_with_retry(prompt, tries=3, **kwargs):
    for attempt in range(tries):
        try:
            return ask(prompt, **kwargs)
        except Exception as error:
            if attempt == tries - 1:
                raise
            wait = 2 ** attempt
            print(f"attempt {attempt + 1} failed ({error}); retrying in {wait} s")
            time.sleep(wait)

In [16]:
real_model, course_llm.MODEL = course_llm.MODEL, "no-such-model"
try:
    ask_with_retry("hello")
except Exception as error:
    print("gave up:", error)
finally:
    course_llm.MODEL = real_model

attempt 1 failed (Ollama has no model 'no-such-model'. Run:  ollama pull no-such-model); retrying in 1 s


attempt 2 failed (Ollama has no model 'no-such-model'. Run:  ollama pull no-such-model); retrying in 2 s


gave up: Ollama has no model 'no-such-model'. Run:  ollama pull no-such-model


Retrying a missing model is pointless, of course: in real code, retry only errors that can heal (timeouts, rate limits, server errors), and fail fast on the rest (wrong key, wrong model name, bad request).

## 6. Keeping keys safe

- Keys live in `.env` at the repo root, which is in `.gitignore`. **Never** type a key into a notebook: notebooks get shared, committed and screenshotted.
- If a key is ever committed or posted, **revoke it** in the provider's console at once; deleting the commit is not enough.
- Give each project its own key with a spending limit, so a leak or a runaway loop has a ceiling.

## Quiz answers

1. Input (prompt) tokens and output (answer) tokens, usually at different prices; output is typically several times more expensive.
2. 1,000 × (150 × 0.10 + 40 × 0.50) / 1,000,000 = $0.035.
3. It is sent with every request, so its tokens are paid for every time.
4. The provider name, the web address (`base_url`), the key and the model name.
5. Timeouts, rate limits (HTTP 429) and server errors (5xx). Not a wrong key or a bad request: those will fail every time.

**Next:** [09.10 Retrieval-augmented generation](../10-rag/)